# 20260919 Visualize

Plot/check one aligned recording at a time. This notebook starts after `20260919_dataframe_construction.ipynb` has an aligned CSV available.

In [ ]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from preprocess_functions.manifest import load_session_records
from preprocess_functions.pipeline import (
    default_aligned_session_path,
    default_cue_events_path,
    default_trials_path,
)
from preprocess_functions import boundary_tuning, plot, roi, segment


In [ ]:
OUTPUT_ROOT = Path("preprocess_out")
MANIFEST = OUTPUT_ROOT / "manifest_with_cells.csv"
if not MANIFEST.exists():
    MANIFEST = OUTPUT_ROOT / "manifest_with_h5.csv"
if not MANIFEST.exists():
    MANIFEST = Path("data_paths/RSC_PPC_Cohort1_paths.xlsx")

SHEET = None
LAB_DRIVE = os.environ.get("LAB_DRIVE_PATH") or None
RECORDING_ID = None
FPS = 30.0


In [ ]:
records = load_session_records(MANIFEST, sheet_name=SHEET, lab_drive=LAB_DRIVE)
records_df = pd.DataFrame([
    {
        "recording_id": record.recording_id,
        "session_id": record.session_id,
        "trial_type": record.trial_type,
        "aligned_csv": str(default_aligned_session_path(OUTPUT_ROOT, record)),
        "cell_csv": str(record.cell_csv) if record.cell_csv else None,
    }
    for record in records
])
records_df


In [ ]:
def choose_record(records, recording_id=None):
    if recording_id is None:
        return records[0]
    for record in records:
        if recording_id in {record.recording_id, record.session_id, record.trial_type, record.mouse_id}:
            return record
    raise ValueError(f"No manifest row matched {recording_id!r}.")

record = choose_record(records, RECORDING_ID)
aligned_csv = default_aligned_session_path(OUTPUT_ROOT, record)
print("recording_id:", record.recording_id)
print("aligned CSV:", aligned_csv)


## Upstream MATLAB Check

Before building aligned sessions, the expected order is: local H5 conversion, R2025b EXTRACT, R2021b ActSort/manualActSort labels, then curated-neuron import.

In [ ]:
print("Expected upstream files for this recording:")
print("local H5:", OUTPUT_ROOT / record.recording_id / "neural" / f"{record.recording_id}_miniscope.h5")
print("R2025b EXTRACT:", OUTPUT_ROOT / record.recording_id / "matlab" / f"{record.recording_id}_precomputed_output.mat")
print("R2021b labels:", OUTPUT_ROOT / record.recording_id / "matlab" / f"{record.recording_id}_precomputed_output_LABELS.mat")
print("curated cell CSV:", record.cell_csv)


## Build One Aligned CSV

Run this command when the aligned CSV does not exist yet. Keep `--only` while checking one recording.

In [ ]:
print(
    "python scripts/build_aligned_sessions.py "
    f"{MANIFEST} --output-root {OUTPUT_ROOT} --only {record.recording_id}"
)


In [ ]:
df = pd.read_csv(aligned_csv)
print(df.shape)
display(df.head())


## Load Trial Metadata

If `20260919_dataframe_construction.ipynb` has segmented trials, this loads the saved trial table with cue/trial metadata.

In [ ]:
trials_csv = default_trials_path(OUTPUT_ROOT, record)
cue_events_csv = default_cue_events_path(OUTPUT_ROOT, record)

if trials_csv.exists():
    trials_df = pd.read_csv(trials_csv)
else:
    trials_df = pd.DataFrame()
    print("No trials CSV found yet:", trials_csv)

if cue_events_csv.exists():
    cue_events_df = pd.read_csv(cue_events_csv)
else:
    cue_events_df = pd.DataFrame()
    print("No cue events CSV found yet:", cue_events_csv)

print("trials CSV:", trials_csv)
display(trials_df.head())
print("cue events CSV:", cue_events_csv)
display(cue_events_df.head())


## Choose Full Session or One Trial

Set `TRIAL_IDX` to an integer to plot one trial, or leave it as `None` to plot the full aligned session.

In [ ]:
TRIAL_IDX = None

if TRIAL_IDX is not None and not trials_df.empty:
    trial_row = trials_df.loc[trials_df["trial_idx"] == TRIAL_IDX].iloc[0]
    start_frame = int(trial_row["start_frame"])
    end_frame = int(trial_row["end_frame"])
    plot_df = df.iloc[start_frame:end_frame + 1].copy()
    print("plotting trial:")
    display(trial_row)
    if "cue_start_utc" in trial_row:
        print("cue window:", trial_row.get("cue_key"), trial_row.get("cue_start_utc"), "to", trial_row.get("cue_end_utc"))
else:
    trial_row = None
    plot_df = df
    print("plotting full session")


## Collect or Load Arena ROIs

If ROI JSON files already exist, they are loaded. If they are missing, OpenCV opens the behavior-video frame so you can draw them. This notebook uses the masks for plotting; dataframe construction is still the place that saves ROI columns for trial segmentation.

In [ ]:
ROI_NAMES = ["arena", "startbox_L", "startbox_R"]

def choose_roi_id(record, roi_names, root=PROJECT_ROOT):
    for candidate in [record.recording_id, record.session_id]:
        if candidate and any(roi.roi_json_path(candidate, name, root=root).exists() for name in roi_names):
            return candidate
    return record.recording_id

ROI_ID = choose_roi_id(record, ROI_NAMES)
SAVE_ROI_FEATURES_TO_ALIGNED_CSV = False

def behavior_video_path(record, df):
    if "beh_vid_path" in df.columns and df["beh_vid_path"].notna().any():
        values = df["beh_vid_path"].dropna().astype(str).str.strip()
        values = values[~values.str.lower().isin(["", "nan", "none", "null"])]
        if not values.empty:
            return Path(values.iloc[0])
    if record.beh_vid is not None:
        return Path(record.beh_vid)
    raise FileNotFoundError("No behavior video path found for ROI selection.")

video_path = behavior_video_path(record, df)
print("behavior video:", video_path)

rois, roi_paths = roi.load_or_collect_named_rois(
    video_path=video_path,
    session_id=ROI_ID,
    roi_names=ROI_NAMES,
    root=PROJECT_ROOT,
    folder_name="arena_rois",
)

height, width = roi.get_video_hw(video_path)
masks = roi.build_roi_masks(rois, height, width)
arena_mask = masks["arena"]

if {"ear_mid_x", "ear_mid_y"}.issubset(df.columns):
    df = roi.add_roi_features(
        df,
        rois,
        ref_x="ear_mid_x",
        ref_y="ear_mid_y",
    )
    df = roi.add_arena_only_column(df)

    if trial_row is None:
        plot_df = df
    else:
        plot_df = df.iloc[start_frame:end_frame + 1].copy()

    if SAVE_ROI_FEATURES_TO_ALIGNED_CSV:
        aligned_csv.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(aligned_csv, index=False)
        print("updated aligned CSV:", aligned_csv)

print("ROI JSONs:")
for name, path in roi_paths.items():
    print(f"  {name}: {path}")
print("mask shape:", arena_mask.shape)


## Plot By Plot Type

These cells use the renamed plotting functions: `plot_trajectory`, `plot_hd_trajectory`, `plot_2d_ratemap`, `plot_hd`, `plot_ebc`, and `plot_cell_summary`.

In [ ]:
plot.plot_trajectory(
    plot_df,
    masks=masks or None,
    x_col="ear_mid_x",
    y_col="ear_mid_y",
    downsample=5,
)


In [ ]:
plot.plot_hd_trajectory(
    plot_df,
    masks=masks or None,
    x_col="ear_mid_x",
    y_col="ear_mid_y",
    angle_col="head_dir_rad",
    downsample=20,
)


In [ ]:
cell_cols = [col for col in plot_df.columns if col.startswith("cell_")]
CELL_COL = cell_cols[0] if cell_cols else None
print("cell for plots:", CELL_COL)


In [ ]:
if CELL_COL is not None:
    plot.plot_2d_ratemap(
        plot_df,
        cell_col=CELL_COL,
        arena_mask=arena_mask,
        bins=20,
    )


In [ ]:
if CELL_COL is not None and "head_dir_rad" in plot_df.columns:
    plot.plot_hd(plot_df, cell_col=CELL_COL, head_dir_col="head_dir_rad", n_bins=36)


In [ ]:
required_for_ebc = {"ear_mid_x", "ear_mid_y", "nose.x", "nose.y", "head_dir_rad"}
if CELL_COL is not None and arena_mask is not None and required_for_ebc.issubset(plot_df.columns):
    plot.plot_ebc(
        plot_df,
        arena_mask=arena_mask,
        cell_col=CELL_COL,
        angle_bins=36,
        distance_bins=20,
        boundary_stride=2,
        frame_stride=1,
        smooth_sigma=(1.0, 1.5),
    )


In [ ]:
if CELL_COL is not None and arena_mask is not None and required_for_ebc.issubset(plot_df.columns):
    plot.plot_cell_summary(
        plot_df,
        cell_col=CELL_COL,
        arena_mask=arena_mask,
        egocentric_smooth_sigma=(1.0, 1.5),
        head_direction_bins=36,
    )
